# 🔥 PINN — Inverse Heat Source Identification (1D) — v4
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/pinn-inverse-heat/blob/main/notebooks/01_inverse_1D_colab.ipynb)

**Problem**: recover unknown source $f(x)$ from sparse noisy measurements:
$$-\frac{d^2T}{dx^2} = f(x), \quad x\in(0,1), \quad T(0)=T(1)=0$$

## Strategy — Sequential 3-phase training

The naive joint training fails because $w_{data} \gg w_{pde}$ starves $\mathcal{N}_f$ of gradient.
We use a **sequential identification** approach instead:

| Phase | What trains | Loss terms | Goal |
|-------|------------|-----------|------|
| 1 | $\mathcal{N}_T$ only | Data + BCs | Get accurate $\hat{T}$ |
| 2 | $\mathcal{N}_f$ only (frozen $\mathcal{N}_T$) | $\|\mathcal{N}_f + \hat{T}''\|^2$ + Tikhonov | Identify $f$ from $-\hat{T}''$ |
| 3 | Both jointly | All terms balanced | Fine-tune |


In [ ]:
# ══════════════════════════════════════════════════
#  CELL 1 — Colab setup
# ══════════════════════════════════════════════════
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repo to get src/ modules
    if not os.path.exists('/content/pinn-inverse-heat'):
        !git clone https://github.com/MaximeAuger/pinn-inverse-heat.git
    sys.path.insert(0, '/content/pinn-inverse-heat/src')
    
    # Create results dir in Colab's writable space
    RESULTS_DIR = '/content/results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running on Colab. Results → {RESULTS_DIR}')
else:
    sys.path.insert(0, '../src')
    RESULTS_DIR = '../results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f'Running locally. Results → {RESULTS_DIR}')

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 2 — Imports
# ══════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

torch.manual_seed(42)
np.random.seed(42)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'PyTorch {torch.__version__}')

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 3 — Architecture
# ══════════════════════════════════════════════════
class MLP(nn.Module):
    def __init__(self, layers, init_scale=1.0):
        super().__init__()
        net = []
        for i in range(len(layers)-1):
            lin = nn.Linear(layers[i], layers[i+1])
            nn.init.xavier_normal_(lin.weight)
            lin.weight.data *= init_scale
            nn.init.zeros_(lin.bias)
            net.append(lin)
            if i < len(layers)-2:
                net.append(nn.Tanh())
        self.net = nn.Sequential(*net)

    def forward(self, x):
        return self.net(x)


def set_requires_grad(model, flag):
    """Freeze or unfreeze a model's parameters."""
    for p in model.parameters():
        p.requires_grad_(flag)


def d2_dx2(model, x):
    """
    Compute d²model/dx² via autograd.
    Returns the second derivative tensor (same shape as x).
    """
    x_  = x.detach().clone().requires_grad_(True)
    y   = model(x_)
    dy  = torch.autograd.grad(y,  x_, torch.ones_like(y),  create_graph=True)[0]
    d2y = torch.autograd.grad(dy, x_, torch.ones_like(dy), create_graph=True)[0]
    return d2y


torch.manual_seed(42)
pinn       = MLP([1, 64, 64, 64, 1], init_scale=1.0)
source_net = MLP([1, 64, 64, 64, 1], init_scale=0.1)
print('Models created ✓')

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 4 — Data
# ══════════════════════════════════════════════════
def f_true(x):   return np.sin(np.pi*x) + 0.5*np.sin(3*np.pi*x)
def T_true(x):   return (1/np.pi**2)*np.sin(np.pi*x) + (0.5/(9*np.pi**2))*np.sin(3*np.pi*x)

N_OBS = 20;  NOISE = 0.01
x_obs_np = np.linspace(0.05, 0.95, N_OBS)
T_clean  = T_true(x_obs_np)
T_obs_np = T_clean + NOISE*np.max(np.abs(T_clean))*np.random.randn(N_OBS)

x_obs    = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1)
T_obs    = torch.tensor(T_obs_np, dtype=torch.float32).unsqueeze(1)
x_col    = torch.linspace(0, 1, 200).unsqueeze(1)
x_eval   = torch.linspace(0, 1, 400).unsqueeze(1)
x_plot   = np.linspace(0, 1, 400)
x0       = torch.zeros(1,1);  x1 = torch.ones(1,1)

print(f'{N_OBS} observations, noise={NOISE*100:.0f}% ✓')

## Phase 1 — Train $\mathcal{N}_T$ on data only

We first get an accurate temperature field **without any PDE constraint**.
This is essentially a regression with boundary conditions.

$$\mathcal{L}_1 = w_{bc}\,\mathcal{L}_{bc} + w_{data}\,\|\mathcal{N}_T(x^{obs}) - T^{obs}\|^2$$

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 5 — Phase 1: fit T from data
# ══════════════════════════════════════════════════
set_requires_grad(pinn, True)
set_requires_grad(source_net, False)   # frozen

opt1 = optim.Adam(pinn.parameters(), lr=1e-3)
sch1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, patience=5000, factor=0.5, min_lr=1e-5)

hist1 = []
PHASE1_EPOCHS = 100000
snapshots_p1 = []

print('Phase 1 — fitting T(x) from data')
for ep in range(1, PHASE1_EPOCHS+1):
    opt1.zero_grad()
    L_bc   = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2
    L_data = torch.mean((pinn(x_obs) - T_obs)**2)
    loss   = 500.0*L_bc + 1000.0*L_data
    loss.backward()
    opt1.step()
    sch1.step(loss)

    hist1.append(loss.item())

    if ep % 500 == 0 or ep == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).squeeze().numpy().copy()
        snapshots_p1.append((ep, x_eval.squeeze().numpy().copy(), T_s))

    if ep % 1000 == 0:
        lr_now = opt1.param_groups[0]['lr']
        print(f'  ep {ep:5d} | loss {loss.item():.3e} | lr {lr_now:.1e}')
        free_memory()

with torch.no_grad():
    T_phase1 = pinn(x_eval).squeeze().numpy().copy()
T_err1 = np.abs(T_phase1 - T_true(x_plot))
print(f'\nPhase 1 done ✓  T L² = {np.mean(T_err1**2)**0.5:.2e}')

: 

In [ ]:
# ── Quick plot of Phase 1 result ──────────────────
plt.figure(figsize=(7,4))
plt.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='$T^*$ analytical')
plt.plot(x_plot, T_phase1, 'r--', lw=2, label='$\mathcal{N}_T$ after Phase 1')
plt.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, label='Obs.')
plt.title(f'Phase 1 result  |  T L² = {np.mean(T_err1**2)**0.5:.2e}', fontsize=12)
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase1_T.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()

## Phase 2 — Identify $f$ from frozen $\mathcal{N}_T$

With $\mathcal{N}_T$ frozen, we compute $-\hat{T}'' = -\frac{d^2\mathcal{N}_T}{dx^2}$ and train $\mathcal{N}_f$ to match it:

$$\mathcal{L}_2 = \underbrace{\|\mathcal{N}_f(x) + \mathcal{N}_T''(x)\|^2}_{\text{PDE: } f = -T''} + w_{reg}\left\|\frac{d\mathcal{N}_f}{dx}\right\|^2$$

This is a **direct inversion** step — we're essentially fitting $\mathcal{N}_f$ to $-\hat{T}''$.

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 6 — Phase 2: identify f from frozen T
# ══════════════════════════════════════════════════
set_requires_grad(pinn, False)          # FREEZE T
set_requires_grad(source_net, True)     # train f only

opt2 = optim.Adam(source_net.parameters(), lr=1e-3)
sch2 = optim.lr_scheduler.ReduceLROnPlateau(opt2, patience=5000, factor=0.5, min_lr=1e-6)

hist2 = []
snapshots_p2 = []
PHASE2_EPOCHS = 100000
W_REG = 1e-3   # Tikhonov — regularise f

# Pre-compute -T'' on collocation grid (no graph needed, used as target)
# We do NOT pre-compute it as a fixed array because pinn is frozen but
# we still need create_graph=False for efficiency.

print('Phase 2 — identifying f(x) = -T\'\'(x)')

for ep in range(1, PHASE2_EPOCHS+1):
    opt2.zero_grad()

    # Compute T'' with frozen T (no gradient w.r.t. pinn params)
    x_c = x_col.detach().clone().requires_grad_(True)
    T_c = pinn(x_c)
    dT  = torch.autograd.grad(T_c,  x_c, torch.ones_like(T_c), create_graph=True)[0]
    d2T = torch.autograd.grad(dT,   x_c, torch.ones_like(dT),  create_graph=False)[0]
    # d2T is now just data (no graph through pinn) → target for f
    target_f = -d2T.detach()   # f* ≈ -T''

    # Train source_net to match target_f
    f_pred = source_net(x_col)
    L_fit  = torch.mean((f_pred - target_f)**2)

    # Tikhonov regularisation on f
    x_r  = x_col.detach().clone().requires_grad_(True)
    f_r  = source_net(x_r)
    df   = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
    L_reg = torch.mean(df**2)

    loss = L_fit + W_REG * L_reg
    loss.backward()
    opt2.step()
    sch2.step(loss)

    hist2.append({'total': loss.item(), 'fit': L_fit.item(), 'reg': L_reg.item()})

    if ep % 500 == 0 or ep == 1:
        with torch.no_grad():
            f_s = source_net(x_eval).squeeze().numpy().copy()
        snapshots_p2.append((ep, x_eval.squeeze().numpy().copy(), f_s))

    if ep % 1000 == 0:
        lr_now = opt2.param_groups[0]['lr']
        print(f'  ep {ep:5d} | L_fit {L_fit.item():.3e} | L_reg {L_reg.item():.3e} | lr {lr_now:.1e}')
        free_memory()

with torch.no_grad():
    f_phase2 = source_net(x_eval).squeeze().numpy().copy()
f_err2 = np.abs(f_phase2 - f_true(x_plot))
print(f'\nPhase 2 done ✓  f L² = {np.mean(f_err2**2)**0.5:.2e}')

In [ ]:
# ── Quick plot of Phase 2 result ──────────────────
plt.figure(figsize=(7,4))
plt.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='$f^*$ true')
plt.plot(x_plot, f_phase2, 'r--', lw=2, label='$\mathcal{N}_f$ after Phase 2')
plt.title(f'Phase 2 result  |  f L² = {np.mean(f_err2**2)**0.5:.2e}', fontsize=12)
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase2_f.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()

## Phase 3 — Joint fine-tuning

Both networks are now well-initialized. We fine-tune jointly with **balanced weights**:

$$\mathcal{L}_3 = \underbrace{w_{pde}\|\mathcal{N}_T'' + \mathcal{N}_f\|^2}_{\text{coupling}} + w_{bc}\,\mathcal{L}_{bc} + w_{data}\,\mathcal{L}_{data} + w_{reg}\,\|\mathcal{N}_f'\|^2$$

Now the weights are **balanced** — `w_pde` is comparable to `w_data` since both networks start from a good solution.

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 7 — Phase 3: joint fine-tuning
# ══════════════════════════════════════════════════
set_requires_grad(pinn, True)
set_requires_grad(source_net, True)

all_params = list(pinn.parameters()) + list(source_net.parameters())
opt3 = optim.Adam(all_params, lr=1e-4)   # lower lr — we're near the solution
sch3 = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=5000, eta_min=1e-6)

hist3 = []
snapshots_p3 = []
PHASE3_EPOCHS = 100000

# Balanced weights
W_PDE  = 10.0
W_BC   = 500.0
W_DATA = 100.0
W_REG  = 1e-7

print('Phase 3 — joint fine-tuning')

for ep in range(1, PHASE3_EPOCHS+1):
    opt3.zero_grad()

    # PDE coupling
    x_c  = x_col.detach().clone().requires_grad_(True)
    T_c  = pinn(x_c)
    dT   = torch.autograd.grad(T_c,  x_c, torch.ones_like(T_c),  create_graph=True)[0]
    d2T  = torch.autograd.grad(dT,   x_c, torch.ones_like(dT),   create_graph=True)[0]
    f_c  = source_net(x_col)
    L_pde = torch.mean((d2T + f_c)**2)

    # BC
    L_bc  = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2

    # Data
    L_data = torch.mean((pinn(x_obs) - T_obs)**2)

    # Tikhonov
    x_r  = x_col.detach().clone().requires_grad_(True)
    f_r  = source_net(x_r)
    df   = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
    L_reg = torch.mean(df**2)

    loss = W_PDE*L_pde + W_BC*L_bc + W_DATA*L_data + W_REG*L_reg
    loss.backward()
    torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
    opt3.step()
    sch3.step()

    hist3.append({'total': loss.item(), 'pde': L_pde.item(),
                  'bc': L_bc.item(), 'data': L_data.item(), 'reg': L_reg.item()})

    if ep % 300 == 0 or ep == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).squeeze().numpy().copy()
            f_s = source_net(x_eval).squeeze().numpy().copy()
        snapshots_p3.append((ep, x_eval.squeeze().numpy().copy(), T_s, f_s))

    if ep % 1000 == 0:
        print(f'  ep {ep:5d} | pde {L_pde.item():.3e} | data {L_data.item():.3e} | '
              f'bc {L_bc.item():.3e}')
        free_memory()

# Final predictions
with torch.no_grad():
    T_final = pinn(x_eval).squeeze().numpy().copy()
    f_final = source_net(x_eval).squeeze().numpy().copy()
snapshots_p3.append(('final', x_eval.squeeze().numpy().copy(), T_final, f_final))

T_err = np.abs(T_final - T_true(x_plot))
f_err = np.abs(f_final - f_true(x_plot))
print(f'\nPhase 3 done ✓')
print(f'  T L² = {np.mean(T_err**2)**0.5:.2e}')
print(f'  f L² = {np.mean(f_err**2)**0.5:.2e}')
free_memory()

## Final Results

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 8 — Final results
# ══════════════════════════════════════════════════
x_np = x_eval.squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='Analytical $T^*$')
ax.plot(x_np, T_final, 'r--', lw=2, label='PINN')
ax.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, alpha=0.8, label='Obs.')
ax.set_title('Temperature $T(x)$', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
T_l2 = np.mean(T_err**2)**0.5
ax.text(0.05, 0.95, f'Max: {T_err.max():.2e}\nL²: {T_l2:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontsize=10)

ax = axes[1]
ax.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='True $f^*$')
ax.plot(x_np, f_final, 'r--', lw=2, label='PINN recovered')
ax.set_title('Source $f(x)$ — recovered', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
f_l2 = np.mean(f_err**2)**0.5
ax.text(0.05, 0.95, f'Max: {f_err.max():.2e}\nL²: {f_l2:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontsize=10)

plt.suptitle('PINN Inverse Problem — Final Results (Sequential Training)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/final_results.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 9 — Loss histories (all 3 phases)
# ══════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].semilogy([h for h in hist1], 'b-', lw=1.2)
axes[0].set_title('Phase 1 — T regression'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss'); axes[0].grid(alpha=0.3)

axes[1].semilogy([h['fit'] for h in hist2], label='Fit $\mathcal{N}_f$ to $-T\'\'$')
axes[1].semilogy([h['reg'] for h in hist2], label='Tikhonov', ls='--')
axes[1].set_title('Phase 2 — f identification'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

for k, ls in [('pde','-'),('bc','--'),('data','-.'),('reg',':')]:
    axes[2].semilogy([h[k] for h in hist3], ls=ls, label=k.upper())
axes[2].set_title('Phase 3 — joint fine-tuning'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Loss History — 3 Phases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_history.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 10 — Training animation (Phase 3)
# ══════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

def animate(i):
    lbl, xs, Ts, fs = snapshots_p3[i]
    for ax in axes: ax.cla()
    axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2, alpha=0.5, label='$T^*$')
    axes[0].plot(xs, Ts, 'r-', lw=2, label='PINN')
    axes[0].scatter(x_obs_np, T_obs_np, c='gray', s=25, zorder=5)
    axes[0].set_ylim(-0.005, 0.14)
    axes[0].set_title(f'T(x) — phase3 ep {lbl}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(x_plot, f_true(x_plot), 'b-', lw=2, alpha=0.5, label='$f^*$')
    axes[1].plot(xs, fs, 'r-', lw=2, label='PINN')
    axes[1].set_ylim(-1.8, 1.8)
    axes[1].set_title(f'f(x) — phase3 ep {lbl}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()

ani = animation.FuncAnimation(fig, animate, frames=len(snapshots_p3), interval=200)
gif_path = f'{RESULTS_DIR}/training_animation.gif'
ani.save(gif_path, writer='pillow', fps=4, dpi=80)
plt.close(); free_memory()
print(f'GIF saved ✓  ({os.path.getsize(gif_path)//1024} KB)')
from IPython.display import Image
Image(gif_path)

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 11 — Noise sensitivity
# ══════════════════════════════════════════════════
def sequential_train(x_obs_t, T_obs_t, ep1=3000, ep2=3000, ep3=2000, w_reg=1e-3):
    """Full sequential pipeline. Returns f_pred (numpy)."""
    torch.manual_seed(0)
    _pinn = MLP([1,64,64,64,1], init_scale=1.0)
    _src  = MLP([1,64,64,64,1], init_scale=0.1)

    # Phase 1
    set_requires_grad(_pinn, True); set_requires_grad(_src, False)
    _opt = optim.Adam(_pinn.parameters(), lr=1e-3)
    for ep in range(ep1):
        _opt.zero_grad()
        loss = 500*(_pinn(x0).squeeze()**2 + _pinn(x1).squeeze()**2) + \
               1000*torch.mean((_pinn(x_obs_t) - T_obs_t)**2)
        loss.backward(); _opt.step()
        if ep % 1000 == 0: free_memory()

    # Phase 2
    set_requires_grad(_pinn, False); set_requires_grad(_src, True)
    _opt2 = optim.Adam(_src.parameters(), lr=1e-3)
    for ep in range(ep2):
        _opt2.zero_grad()
        x_c = x_col.detach().clone().requires_grad_(True)
        T_c = _pinn(x_c)
        dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
        d2T = torch.autograd.grad(dT,  x_c, torch.ones_like(dT),  create_graph=False)[0]
        tgt = -d2T.detach()
        f_p = _src(x_col)
        x_r = x_col.detach().clone().requires_grad_(True)
        f_r = _src(x_r)
        df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
        loss = torch.mean((f_p - tgt)**2) + w_reg*torch.mean(df**2)
        loss.backward(); _opt2.step()
        if ep % 1000 == 0: free_memory()

    # Phase 3
    set_requires_grad(_pinn, True); set_requires_grad(_src, True)
    _params = list(_pinn.parameters()) + list(_src.parameters())
    _opt3 = optim.Adam(_params, lr=1e-4)
    for ep in range(ep3):
        _opt3.zero_grad()
        x_c = x_col.detach().clone().requires_grad_(True)
        T_c = _pinn(x_c)
        dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
        d2T = torch.autograd.grad(dT,  x_c, torch.ones_like(dT),  create_graph=True)[0]
        f_c = _src(x_col)
        L_pde  = torch.mean((d2T + f_c)**2)
        L_bc   = _pinn(x0).squeeze()**2 + _pinn(x1).squeeze()**2
        L_data = torch.mean((_pinn(x_obs_t) - T_obs_t)**2)
        x_r = x_col.detach().clone().requires_grad_(True)
        f_r = _src(x_r)
        df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
        L_reg = torch.mean(df**2)
        loss = 10*L_pde + 500*L_bc + 100*L_data + w_reg*L_reg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(_params, 1.0)
        _opt3.step()
        if ep % 1000 == 0: free_memory()

    with torch.no_grad():
        f_pred = _src(x_eval).squeeze().numpy().copy()
    del _pinn, _src, _params
    free_memory()
    return f_pred


noise_levels = [0.0, 0.01, 0.02, 0.05, 0.10]
results_noise = {}
x_np = x_eval.squeeze().numpy()

for noise in noise_levels:
    np.random.seed(0)
    T_n   = T_clean + noise*np.max(np.abs(T_clean))*np.random.randn(N_OBS)
    T_n_t = torch.tensor(T_n, dtype=torch.float32).unsqueeze(1)
    f_pred = sequential_train(x_obs, T_n_t)
    l2 = float(np.mean((f_pred - f_true(x_np))**2)**0.5)
    results_noise[noise] = (f_pred, l2)
    print(f'  noise={noise*100:5.1f}%  →  f L² = {l2:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.plasma
axes[0].plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$', zorder=10)
for i, (n, (fp, _)) in enumerate(results_noise.items()):
    axes[0].plot(x_np, fp, '--', lw=1.5, color=cmap(i/len(noise_levels)), label=f'{n*100:.0f}%')
axes[0].set_title('Source recovery vs noise'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([n*100 for n in noise_levels], [results_noise[n][1] for n in noise_levels], 'ro-', ms=8)
axes[1].set_xlabel('Noise (%)'); axes[1].set_ylabel('L² error'); axes[1].grid(alpha=0.3)
axes[1].set_title('L² error vs noise')
plt.suptitle('Noise Sensitivity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/noise_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 12 — Tikhonov study
# ══════════════════════════════════════════════════
reg_weights = [0.0, 1e-4, 1e-3, 1e-2]
results_reg = {}
np.random.seed(0)
T_noisy   = T_clean + 0.03*np.max(np.abs(T_clean))*np.random.randn(N_OBS)
T_noisy_t = torch.tensor(T_noisy, dtype=torch.float32).unsqueeze(1)

for wr in reg_weights:
    f_pred = sequential_train(x_obs, T_noisy_t, w_reg=wr)
    l2 = float(np.mean((f_pred - f_true(x_np))**2)**0.5)
    results_reg[wr] = (f_pred, l2)
    print(f'  w_reg={wr:.0e}  →  f L² = {l2:.4f}')

plt.figure(figsize=(9, 5))
plt.plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$ true')
cmap2 = plt.cm.viridis
for i, (wr, (fp, l2)) in enumerate(results_reg.items()):
    plt.plot(x_np, fp, '--', lw=1.8, color=cmap2(i/len(reg_weights)),
             label=f'$w_{{reg}}={wr:.0e}$ (L²={l2:.3f})')
plt.title('Effect of Tikhonov Regularization (noise=3%)', fontsize=12)
plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tikhonov_study.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# ══════════════════════════════════════════════════
#  CELL 13 — Summary
# ══════════════════════════════════════════════════
import glob
print('═'*50)
print('  RESULTS SUMMARY')
print('═'*50)
print(f'  Temperature  L² error : {np.mean(T_err**2)**0.5:.2e}')
print(f'  Source       L² error : {np.mean(f_err**2)**0.5:.2e}')
print(f'  Observations           : {N_OBS} pts, noise={NOISE*100:.0f}%')
print(f'  Training phases        : Phase1={5000} | Phase2={5000} | Phase3={3000}')
print('  Strategy               : Sequential identification')
print('═'*50)
print('Generated files:')
for fp in sorted(glob.glob(f'{RESULTS_DIR}/*')):
    print(f'  {os.path.basename(fp):40s} {os.path.getsize(fp)//1024:>5} KB')
if IN_COLAB:
    print('\nSave to Drive:')
    print('  from google.colab import drive; drive.mount("/content/drive")')
    print('  !cp -r /content/results /content/drive/MyDrive/')